# 🎥 VideoDB Multicam Quickstart
---

Welcome to the **VideoDB Multicam Quickstart**! Learn how to build professional multi-camera surveillance systems with real-time AI event detection and synchronized multi-angle composition.

### 🎯 What You'll Build

A complete AI-powered multicam surveillance system that demonstrates:

- **👁️ SEE**: Connect to 4 synchronized security cameras and preview live feeds
- **🧠 UNDERSTAND**: Index visual content across all cameras with AI analysis
- **🎬 ACT**: Detect events in real-time, receive WebSocket alerts, and create synchronized multi-angle videos

By the end, you'll have a working system that:
- Monitors 4 cameras simultaneously
- Detects specific events (people with trolley bags, crowd formation, unattended luggage)
- Sends real-time alerts via WebSocket
- Creates synchronized 2x2 grid videos for evidence review

---

## 🛠 Setup & Installation

Let's start by installing the VideoDB Python SDK and connecting to your account.

---

### 📦 Install VideoDB

VideoDB is available as a [Python package](https://pypi.org/project/videodb). Run the cell below to install it.

In [ ]:
!pip install -q videodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 1.4 MB/s eta 0:00:00


### 🔗 Connect to VideoDB

You'll need an API key to interact with VideoDB. Provide it securely when prompted below.

> 💡 **Tip:** Get your free API key from the [VideoDB Console](https://console.videodb.io). Get $20 in free API credits upon sign-up — no credit card required!

In [ ]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 👁️ SEE - Connect Multi-Camera System

First, we'll connect to **4 synchronized security cameras** monitoring a public plaza. VideoDB handles RTSP stream ingestion seamlessly.

---

### 📹 Configure Cameras

In [ ]:
# Multi-camera configuration
CAMERA_CONFIG = {
    "cam1": {"name": "Plaza Overview", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam1"},
    "cam2": {"name": "Main Walkway", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam2"},
    "cam3": {"name": "Stairway Junction", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam3"},
    "cam4": {"name": "Central Plaza", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam4"},
}

print("📋 Camera Configuration:")
for cam_id, info in CAMERA_CONFIG.items():
    print(f"   {cam_id}: {info['name']}")

📋 Camera Configuration:
   cam1: Plaza Overview
   cam2: Main Walkway
   cam3: Stairway Junction
   cam4: Central Plaza


---
### Connect all streams

In [ ]:
# Connect all cameras
print("🔌 Connecting to all cameras...\n")

streams = {}
for cam_id, cam_info in CAMERA_CONFIG.items():
    stream = coll.connect_rtstream(
        name=f"Surveillance_{cam_id}",
        url=cam_info["url"],
        store=True,  # enables recording storage
    )
    streams[cam_id] = {"stream": stream, "info": cam_info}
    print(f"✅ {cam_id}: {stream.id}")

print(f"\n✅ All {len(streams)} cameras connected!")

🔌 Connecting to all cameras...

✅ cam1: rts-019d68b3-07bb-79a1-b6ff-5f7ce8e885c7
✅ cam2: rts-019d68b3-0981-7652-8b12-477194a8b1c5
✅ cam3: rts-019d68b3-0b3a-7b83-8cc9-c87188a4b599
✅ cam4: rts-019d68b3-0d1f-7590-85a0-b485ab44383b

✅ All 4 cameras connected!


#### To reconnect to existing stream:

In [ ]:
# EXISTING_STREAM_IDS = {
#     "cam1": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 stream ID
#     "cam2": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 stream ID
#     "cam3": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 stream ID
#     "cam4": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam4 stream ID
# }

# print("🔌 Reconnecting to existing cameras...\n")

# streams = {}
# for cam_id, rtstream_id in EXISTING_STREAM_IDS.items():
#     stream = coll.get_rtstream(rtstream_id)
#     streams[cam_id] = {"stream": stream, "info": CAMERA_CONFIG[cam_id]}
#     print(f"✅ {cam_id}: {stream.id} ({CAMERA_CONFIG[cam_id]['name']})")

# print(f"\n✅ All {len(streams)} cameras reconnected!")

🔌 Reconnecting to existing cameras...

✅ cam1: rts-019d6850-dbf7-73a1-9d43-1a1f156b5a40 (Plaza Overview)
✅ cam2: rts-019d6850-dd25-7f00-8028-cd31ac9250f6 (Main Walkway)
✅ cam3: rts-019d6850-de8d-77f3-92df-98ea17b67771 (Stairway Junction)
✅ cam4: rts-019d6850-dfad-7193-854b-386d69c3fe65 (Central Plaza)

✅ All 4 cameras reconnected!


---

### 📺 Preview Live Feeds

Let's preview the last 2 minutes from each camera to verify connectivity:
> Wait for atleast 2 minuts before executing the following cell.

In [ ]:
from IPython.display import HTML
import time

# Get timestamps for last 2 minutes
now = int(time.time())
ten_seconds_ago = now - 120

print("📹 Generating preview streams for all cameras...\n")

# Generate streams and collect stream URLs
stream_urls = {}
video_titles = []
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream preview)
    player_url = stream.generate_stream(ten_seconds_ago, now)

    stream_url = stream.stream_url  # Get actual stream URL
    stream_urls[cam_id] = stream_url
    video_titles.append(cam_data['info']['name'])
    print(f"✅ {cam_data['info']['name']}")
    print(f"  {player_url}\n")

print("\n📺 Displaying all 4 cameras in 2x2 grid:\n")

# Create HTML with 4 videos in 2x2 grid
html_content = """
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; max-width: 720px;">
"""

# Populate HTML with iframes for each video
for i, (cam_id, url) in enumerate(stream_urls.items()):
    html_content += f'''
        <div style="text-align: center;">
            <h4>{video_titles[i]}</h4>
            <iframe src="{url}" width="100%" height="200" frameborder="0" allowfullscreen></iframe>
        </div>
    '''

html_content += '</div>'

# Display all 4 videos at once
display(HTML(html_content))

📹 Generating preview streams for all cameras...

✅ Plaza Overview
  https://player.videodb.io/watch?v=uUcRN3Atexs

✅ Main Walkway
  https://player.videodb.io/watch?v=NIqwtkJLkng

✅ Stairway Junction
  https://player.videodb.io/watch?v=TLW_D3GfsYA

✅ Central Plaza
  https://player.videodb.io/watch?v=x2eWq-HY9VE


📺 Displaying all 4 cameras in 2x2 grid:



---

## 🧠 UNDERSTAND - AI-Powered Visual Analysis

Now we'll set up AI-powered visual indexing across all 4 cameras. The AI will analyze frames every 10 seconds to detect people, objects, and unusual activity.

---

### 🔍 Configure Visual Indexing

In [ ]:
# Scene indexing configuration
SCENE_INDEX_CONFIG = {
    "batch_config": {
        "type": "time",
        "value": 10,  # Analyze every 10 seconds
        "frame_count": 1
    },
    "prompt": """Analyze this surveillance footage. Describe:
    1. People with trolley bags, backpacks, or luggage
    2. Crowd behavior (groups gathering, movement patterns)
    3. Unusual objects (unattended items)
    Be specific about location and appearance."""
}

print("📋 Visual Index Configuration:")
print(f"   Analyzing every {SCENE_INDEX_CONFIG['batch_config']['value']} seconds")
print(f"   AI Focus: People, luggage, crowd behavior")

📋 Visual Index Configuration:
   Analyzing every 10 seconds
   AI Focus: People, luggage, crowd behavior


In [ ]:
# Index visuals for all cameras
print("🔧 Creating visual indexes...\n")

scene_indexes = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    scene_index = stream.index_visuals(
        batch_config=SCENE_INDEX_CONFIG["batch_config"],
        prompt=SCENE_INDEX_CONFIG["prompt"],
        name=f"Surveillance_{cam_id}_Index"
    )
    scene_indexes[cam_id] = {
        "index": scene_index,
        "index_id": scene_index.rtstream_index_id
    }
    print(f"✅ {cam_id}: {scene_index.rtstream_index_id}")

print(f"\n✅ Visual indexing active on all {len(scene_indexes)} cameras!")

🔧 Creating visual indexes...

✅ cam1: d54224ba12dec6fd
✅ cam2: d8f48f9ce22a176e
✅ cam3: 72f0d8315ca9dc07
✅ cam4: 01105faa886ad7a0

✅ Visual indexing active on all 4 cameras!


#### If you've already created scene indexes, you can reconnect to them instead of creating new ones.

In [ ]:
# EXISTING_INDEX_IDS = {
#     "cam1": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 index ID
#     "cam2": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 index ID
#     "cam3": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 index ID
#     "cam4": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam4 index ID
# }
#
# print("🔧 Reconnecting to existing scene indexes...\n")
#
# scene_indexes = {}
# for cam_id, index_id in EXISTING_INDEX_IDS.items():
#     stream = streams[cam_id]["stream"]
#     scene_index = stream.get_scene_index(index_id)
#     scene_indexes[cam_id] = {
#         "index": scene_index,
#         "index_id": scene_index.rtstream_index_id
#     }
#     print(f"✅ {cam_id}: {scene_index.rtstream_index_id}")
#
# print(f"\n✅ Reconnected to all {len(scene_indexes)} scene indexes!")

🔧 Reconnecting to existing scene indexes...

✅ cam1: ddf627175745bac2
✅ cam2: 970bcb6384b6a9e7
✅ cam3: 8e21cb5285ad6493
✅ cam4: 64b38c0104ecf9fd

✅ Reconnected to all 4 scene indexes!


---

### 👀 Review Indexed Scenes

Let's see what the AI has detected so far. We'll look at scenes from two different camera angles:

In [ ]:
def display_scenes(cam_id, num_scenes=3, max_chars=200):
    """Display indexed scenes from a specific camera (compact format)"""
    cam_name = streams[cam_id]["info"]["name"]
    scene_index = scene_indexes[cam_id]["index"]

    print(f"📹 {cam_name}:")

    scenes_data = scene_index.get_scenes(page_size=num_scenes)

    if scenes_data and scenes_data.get("scenes"):
        for i, scene in enumerate(scenes_data.get("scenes"), 1):
            start = scene.get("start", 0)
            description = scene.get("description", "No description")

            # Clean up description: remove extra whitespace and newlines
            clean_desc = " ".join(description.split())

            # Truncate if too long
            if len(clean_desc) > max_chars:
                clean_desc = clean_desc[:max_chars].rsplit(' ', 1)[0] + "..."

            print(f"  └─ Scene {i} [{int(start)}s]: {clean_desc}")
        print()  # Single newline after all scenes
    else:
        print("  └─ No scenes indexed yet. Wait a few moments.\n")

---
#### 👀 Cam 1 : Plaza Overview

In [ ]:
# Camera 1 - Plaza Overview
display_scenes("cam1")

# Camera 2 - Main Walkway
display_scenes("cam2")

# Camera 3 - Stairway Junction
display_scenes("cam3")

# Camera 4 - Central Plaza
display_scenes("cam4")

📹 Plaza Overview:
  └─ Scene 1 [1775578574s]: Here's an analysis of the surveillance footage: **1. People with trolley bags, backpacks, or luggage:** * **Trolley Bags/Luggage:** * **Mid-left:** One person in a light green/khaki jacket and dark...
  └─ Scene 2 [1775578561s]: Here's an analysis of the surveillance footage: **1. People with trolley bags, backpacks, or luggage:** * **Trolley Bags/Luggage:** * **Mid-left, near the red van:** A person wearing a dark jacket...
  └─ Scene 3 [1775578549s]: Here's an analysis of the surveillance footage: **1. People with trolley bags, backpacks, or luggage:** * **Trolley Bags/Luggage:** * **Left-center foreground:** A black, upright trolley bag is...

📹 Main Walkway:
  └─ Scene 1 [1775578570s]: Here's an analysis of the surveillance footage: **1. People with trolley bags, backpacks, or luggage:** * **Trolley Bag:** One individual, located in the mid-ground towards the right side of the...
  └─ Scene 2 [1775578556s]: Here's an analysis of the sur

💡 **Multi-angle coverage is active!** All 4 cameras are continuously analyzing and understanding what's happening in the plaza.

---

---

## 🎬 ACT - Event Detection & Multi-Angle Composition

Now comes the exciting part! We'll:
1. Create event detection rules
2. Monitor for alerts in real-time via WebSocket
3. Create synchronized multi-camera evidence videos

---

### 🚨 Define Events to Detect

In [ ]:
# Define 3 surveillance events
EVENTS_CONFIG = [
    {"label": "person_with_trolley", "prompt": "Detect any person with a trolley bag or rolling suitcase."},
    {"label": "large_crowd_formation", "prompt": "Identify 5+ people gathering quickly."},
    {"label": "unattended_luggage", "prompt": "Detect luggage left unattended for 1+ minute."},
]

print("🎯 Creating event detection rules...\n")

events = {}
for cfg in EVENTS_CONFIG:
    event_id = conn.create_event(
        event_prompt=cfg["prompt"],
        label=cfg["label"]
    )
    events[cfg["label"]] = {"event_id": event_id}
    print(f"Event: {cfg['label']}")
    print(f"    ID: {event_id}")

print(f"\n✅ {len(events)} events ready for detection!")

🎯 Creating event detection rules...

Event: person_with_trolley
    ID: bc04500633ce597d
Event: large_crowd_formation
    ID: 453bdc10df4be372
Event: unattended_luggage
    ID: 310a4797c7bc84f8

✅ 3 events ready for detection!


---

### 🔌 Connect WebSocket for Real-Time Alerts

WebSockets let us receive alerts instantly as they happen:

In [ ]:
import asyncio

# Connect to WebSocket
ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()

print(f"✅ WebSocket connected!")
print(f"   Connection ID: {ws.connection_id}")

INFO:videodb.websocket_client:WebSocket connected with ID: bdRjPfJ2vHcCHFA=


✅ WebSocket connected!
   Connection ID: bdRjPfJ2vHcCHFA=


Create Alerts to recieve on the web0socket.

In [ ]:
# Create alerts for all cameras × all events
print("🔔 Creating alerts...\n")

alerts = {}
for cam_id, idx_data in scene_indexes.items():
    alerts[cam_id] = {}
    print(f"--- Alerts for Camera: {cam_id} ---\n")
    for label, evt in events.items():
        alert_id = idx_data["index"].create_alert(
            evt["event_id"],
            callback_url="",  # Empty = WebSocket only
            ws_connection_id=ws.connection_id
        )
        alerts[cam_id][label] = alert_id
        print(f"Alert: {label}")
        print(f"    ID: {alert_id}")
    print(f"\n")

total_alerts = len(alerts) * len(events)
print(f"\n✅ {total_alerts} alerts active (4 cameras × 3 events)")



🔔 Creating alerts...

--- Alerts for Camera: cam1 ---

Alert: person_with_trolley
    ID: 9b4ab9a9b11175ae
Alert: large_crowd_formation
    ID: bb2bdd013805c672
Alert: unattended_luggage
    ID: 59004bd020be28c6


--- Alerts for Camera: cam2 ---

Alert: person_with_trolley
    ID: 9a95916c7c27c799
Alert: large_crowd_formation
    ID: c447921cc5ed4d57
Alert: unattended_luggage
    ID: f15471aa48d7db36


--- Alerts for Camera: cam3 ---

Alert: person_with_trolley
    ID: 68a6c40c7f357cd5
Alert: large_crowd_formation
    ID: 75a23f544147e775
Alert: unattended_luggage
    ID: 39d59f83598010d2


--- Alerts for Camera: cam4 ---

Alert: person_with_trolley
    ID: c14e78d0be2df1ac
Alert: large_crowd_formation
    ID: 5c1deda88f6231e1
Alert: unattended_luggage
    ID: 6a575b86e255b3cc



✅ 12 alerts active (4 cameras × 3 events)


---

### 👂 Listen for Alerts

Let's listen for 30 seconds. When events are detected, we'll receive instant notifications:

In [ ]:
# Create mapping from rtstream_id to cam info for easy lookup
rtstream_to_cam = {
    streams[cam_id]["stream"].id: {"cam_id": cam_id, "cam_name": CAMERA_CONFIG[cam_id]["name"]}
    for cam_id in streams.keys()
}

# Store alerts for later analysis
received_alerts = []

async def listen_for_alerts():
    timeout = 30  # Adjustable
    print(f"⏱️ Listening for alerts ({timeout} seconds)...")
    print("💡 The surveillance system will trigger alerts when events are detected\n")

    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if msg.get("channel") == "alert":
                    received_alerts.append(msg)
                    alert_num = len(received_alerts)
                    data = msg.get("data", {})

                    # Get camera info
                    rtstream_id = msg.get("rtstream_id", "unknown")
                    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})

                    print(f"\n🚨 ALERT #{alert_num} RECEIVED!")
                    print(f"   Camera: {cam_info['cam_id']} - {cam_info['cam_name']}")
                    print(f"   Event: {data.get('label', 'N/A')}")
                    print(f"   Time: {msg.get('timestamp', 'N/A')}")
                    print(f"   Confidence: {data.get('confidence', 'N/A')}")
                    print(f"   📹 Clip URL: {data.get('stream_url', 'N/A')}")
                    print(f"   💡 Explanation: {data.get('explanation', 'N/A')[:150]}...")  # Truncate explanation

    except asyncio.TimeoutError:
        print(f"\n✅ Listening complete! {len(received_alerts)} alert(s) received")

# Start listening
await listen_for_alerts()

⏱️ Listening for alerts (30 seconds)...
💡 The surveillance system will trigger alerts when events are detected


🚨 ALERT #1 RECEIVED!
   Camera: cam3 - Stairway Junction
   Event: person_with_trolley
   Time: 2026-04-07T16:27:20.574581+00:00
   Confidence: 0.95
   📹 Clip URL: https://rt.stream.videodb.io/manifests/rts-019d68b3-0b3a-7b83-8cc9-c87188a4b599/1775579203000000-1775579214000000.m3u8
   💡 Explanation: The scene analysis explicitly identifies two individuals pulling 'trolley bags', which directly matches the alert context for detecting persons with t...

🚨 ALERT #2 RECEIVED!
   Camera: cam1 - Plaza Overview
   Event: person_with_trolley
   Time: 2026-04-07T16:27:21.794710+00:00
   Confidence: 0.95
   📹 Clip URL: https://rt.stream.videodb.io/manifests/rts-019d68b3-07bb-79a1-b6ff-5f7ce8e885c7/1775579206000000-1775579218000000.m3u8
   💡 Explanation: The scene analysis explicitly mentions an individual pulling a 'black trolley bag', which matches the alert context for detecting a p

---

### 📊 Analyze Alert Metrics

In [ ]:
# Count alerts by event type
by_type = {}
for alert in received_alerts:
    label = alert.get("data", {}).get("label", "unknown")
    by_type[label] = by_type.get(label, 0) + 1

print("📊 Alerts by Event Type:")
print("-" * 50)
for label, count in sorted(by_type.items(), key=lambda x: x[1], reverse=True):
    # Format label: person_with_trolley -> Person with Trolley
    formatted_label = label.replace("_", " ").title()
    print(f"   {formatted_label:.<35} {count}")

# Count alerts by camera (using rtstream_to_cam mapping)
by_cam = {}
for alert in received_alerts:
    rtstream_id = alert.get("rtstream_id", "unknown")
    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})
    cam_key = f"{cam_info['cam_id']} - {cam_info['cam_name']}"
    by_cam[cam_key] = by_cam.get(cam_key, 0) + 1

print("\n📹 Alerts by Camera:")
print("-" * 50)
for cam, count in sorted(by_cam.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cam:.<35} {count}")

print(f"\n✅ Total: {len(received_alerts)} alerts received")

📊 Alerts by Event Type:
--------------------------------------------------
   Person With Trolley................ 4
   Large Crowd Formation.............. 1

📹 Alerts by Camera:
--------------------------------------------------
   cam3 - Stairway Junction........... 2
   cam4 - Central Plaza............... 2
   cam1 - Plaza Overview.............. 1

✅ Total: 5 alerts received


---

### 🔍 Select Alert to Investigate

Let's investigate a "person with trolley" alert and create a multi-angle evidence video:

In [ ]:
# Auto-select first trolley alert
selected = next(
    (a for a in received_alerts if a["data"]["label"] == "person_with_trolley"),
    None
)

if selected:
    data = selected["data"]
    print(f"🚨 Selected Alert: {data['label']}")
    print(f"   Confidence: {data['confidence']}")
    print(f"   Time: {selected.get('timestamp')}")
else:
    print("⚠️ No trolley alerts received. Try running for longer or select different event.")

🚨 Selected Alert: person_with_trolley
   Confidence: 0.95
   Time: 2026-04-07T16:27:20.574581+00:00


---

### 🎥 Generate Synchronized Multi-Camera Clips

Now we'll extract the same moment from all 4 cameras:

In [ ]:
# Extract timestamp from alert
data = selected["data"]
start_time = data.get("start")  # Unix timestamp
end_time = data.get("end")

# Add 10 second padding for context
PADDING = 10
clip_start = int(start_time - PADDING)
clip_end = int(end_time + PADDING)
clip_duration = clip_end - clip_start

print(f"⏱️ Alert Time Window: {start_time} → {end_time}")
print(f"📹 With padding: {clip_start} → {clip_end}\n")

# Generate synchronized streams for all cameras
stream_urls = {}
player_urls_map = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream)
    player_url = stream.generate_stream(clip_start, clip_end)
    stream_url = stream.stream_url

    stream_urls[cam_id] = stream_url
    player_urls_map[cam_id] = player_url

print(f"✅ 4 synchronized clips ready\n")

print("Player URLs for each camera:")
for cam_id, p_url in player_urls_map.items():
    cam_name = CAMERA_CONFIG[cam_id]["name"]
    print(f"{cam_id} : {cam_name}")
    print(f"    {p_url}")

⏱️ Alert Time Window: 1775579203.9198503 → 1775579213.9479632
📹 With padding: 1775579193 → 1775579223

✅ 4 synchronized clips ready

Player URLs for each camera:
cam1 : Plaza Overview
    https://player.videodb.io/watch?v=G7JkfEvJ7YA
cam2 : Main Walkway
    https://player.videodb.io/watch?v=EPzM5n1vCbU
cam3 : Stairway Junction
    https://player.videodb.io/watch?v=B2XJMv1VXlo
cam4 : Central Plaza
    https://player.videodb.io/watch?v=96A52arFHug


---

### 💾 Download Clips for Composition

Since the Timeline Editor requires VideoDB media IDs, we'll download these clips using ffmpeg.

In [ ]:
import subprocess
import os

# Download all 4 clips using ffmpeg
print("📥 Downloading clips with ffmpeg...\n")

downloads = {}
for cam_id, stream_url in stream_urls.items():
    output_file = f"{cam_id}_clip.mp4"

    # Use ffmpeg to download HLS stream
    cmd = [
        'ffmpeg',
        '-y',  # Overwrite output file
        '-i', stream_url,  # Input HLS URL
        '-c', 'copy',  # Copy codec (no re-encoding)
        '-loglevel', 'error',  # Only show errors
        output_file
    ]

    print(f"   {cam_id}: Downloading...")
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        downloads[cam_id] = {"name": output_file}
        print(f"   ✅ {cam_id}: Saved as {output_file}\n")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ {cam_id}: Failed - {e.stderr.decode()}")
        downloads[cam_id] = {"name": None, "error": str(e)}

print(f"✅ Downloaded {len([d for d in downloads.values() if d.get('name')])} clips!")

📥 Downloading clips with ffmpeg...

   cam1: Downloading...
   ✅ cam1: Saved as cam1_clip.mp4

   cam2: Downloading...
   ✅ cam2: Saved as cam2_clip.mp4

   cam3: Downloading...
   ✅ cam3: Saved as cam3_clip.mp4

   cam4: Downloading...
   ✅ cam4: Saved as cam4_clip.mp4

✅ Downloaded 4 clips!


#### 💾 Uploading back to VideoDB

In [ ]:
# Upload clips back to VideoDB
print("📤 Uploading clips to VideoDB...\n")

videos = {}
for cam_id, dl_info in downloads.items():
    video = coll.upload(file_path=dl_info["name"])
    videos[cam_id] = {"video": video, "video_id": video.id}
    print(f"✅ {cam_id}: {video.id}\n")

print("✅ All clips uploaded and ready for composition!")

📤 Uploading clips to VideoDB...

✅ cam1: m-z-019d68d4-8dd3-7fa1-a4fb-a67d0bff7e5a

✅ cam2: m-z-019d68d4-b9c2-7e52-8aa0-9cd7500c8185

✅ cam3: m-z-019d68d4-e74e-7680-93eb-c04d57fd9e45

✅ cam4: m-z-019d68d5-139a-7560-9153-397711a6a421

✅ All clips uploaded and ready for composition!


---

### 🎬 Create 2x2 Multi-Camera Grid

Now for the exciting part — composing all 4 cameras into a synchronized 2x2 grid! Think of it like layers in a video editor:

- **Timeline**: The canvas
- **Track**: One layer (each camera gets its own track)
- **Clip**: The video positioned in a specific quadrant
- **Asset**: The video media itself

In [ ]:
from videodb.editor import (
    Timeline, Track, Clip, VideoAsset, Position, Offset, Fit,
    TextAsset, Font, Border, Shadow, Background, TextAlignment
)

# Get minimum duration
print("📏 Checking video durations...\n")
video_durations = {}
for cam_id, vid_info in videos.items():
    video = vid_info["video"]
    video_durations[cam_id] = video.length

final_duration = min(video_durations.values())

# Create timeline with medium gray background
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#404040"  # Gray for padding areas

# Define camera positions with offsets for symmetrical padding
cam_configs = [
    {"cam_id": "cam1", "position": Position.top_left, "offset": Offset(x=0.03, y=0.025)},
    {"cam_id": "cam2", "position": Position.top_right, "offset": Offset(x=-0.03, y=0.025)},
    {"cam_id": "cam3", "position": Position.bottom_left, "offset": Offset(x=0.03, y=-0.025)},
    {"cam_id": "cam4", "position": Position.bottom_right, "offset": Offset(x=-0.03, y=-0.025)},
]

print("🎬 Building multi-camera grid...\n")

# Add video tracks
for config in cam_configs:
    cam_id = config["cam_id"]
    clip = Clip(
        asset=VideoAsset(id=videos[cam_id]["video_id"]),
        duration=final_duration,
        fit=Fit.crop,
        position=config["position"],
        offset=config["offset"],
        scale=0.45,  # Larger videos with small padding
    )
    track = Track()
    track.add_clip(0, clip)
    timeline.add_track(track)

# Define label positions - positioned inside each video at top
label_configs = [
    {"cam_id": "cam1", "offset": Offset(x=-0.355, y=-0.45)},
    {"cam_id": "cam2", "offset": Offset(x=0.135, y=-0.45)},
    {"cam_id": "cam3", "offset": Offset(x=-0.355, y=0.05)},
    {"cam_id": "cam4", "offset": Offset(x=0.135, y=0.05)},
]

# Add camera labels
for config in label_configs:
    cam_id = config["cam_id"]
    cam_name = CAMERA_CONFIG[cam_id]["name"]

    label_text_content = f"{cam_id.upper()}: {cam_name}"

    label_text = TextAsset(
        text=label_text_content,
        font=Font(
            family="Clear Sans",
            size=24,
            color="#FFFFFF",
        ),
        background=Background(
          color="#000000",
          height=50,
          width=300,
          text_alignment=TextAlignment.center,
        ),
    )

    label_clip = Clip(
        asset=label_text,
        duration=final_duration,
        offset=config["offset"],
    )

    track = Track()
    track.add_clip(0, label_clip)
    timeline.add_track(track)

print(f"✅ Production-ready multi-camera grid complete!\n")
print(f"   Resolution: 1280x720")
print(f"   Duration: {final_duration:.2f}s")
print(f"   Layout: 2x2 grid with symmetrical padding")

📏 Checking video durations...

🎬 Building multi-camera grid...

✅ Production-ready multi-camera grid complete!

   Resolution: 1280x720
   Duration: 15.00s
   Layout: 2x2 grid with symmetrical padding


In [ ]:
# Generate final multi-camera view
from videodb import play_stream

print("🎬 Generating final video...\n")

stream = timeline.generate_stream()

print("✅ Multi-camera synchronized view ready!")
print(f"Stream: {stream}\n")

play_stream(stream)

🎬 Generating final video...

✅ Multi-camera synchronized view ready!
Stream: https://play.videodb.io/v1/e86d3db9-c3c8-403c-84a4-e5b9a41a8e29.m3u8



<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> You've created a professional synchronized multi-camera surveillance clip — showing the same moment from 4 different angles!
</div>

---

## 🧹 Cleanup

When you're done, let's disconnect all resources:

In [ ]:
print("🧹 Cleaning up resources...\n")

# Close WebSocket
ws_wrapper.close()
print("✅ WebSocket closed")

# Disable all alerts
for cam_id, cam_alerts in alerts.items():
    for label, alert_id in cam_alerts.items():
        try:
            scene_indexes[cam_id]["index"].disable_alert(alert_id)
        except Exception as e:
            pass

print("✅ All alerts disabled")

# Stop all streams
for cam_id, cam_data in streams.items():
    try:
        cam_data["stream"].stop()
    except Exception as e:
        pass

print("✅ All streams stopped")
print("\n🎉 Cleanup complete!")

🧹 Cleaning up resources...

✅ WebSocket closed


/tmp/ipykernel_6026/4006229174.py:4: RuntimeWarning: coroutine 'WebSocketConnection.close' was never awaited
  ws_wrapper.close()


✅ All alerts disabled
✅ All streams stopped

🎉 Cleanup complete!


---

# 🏁 Conclusion: Professional Multicam Surveillance

Congratulations! You've built a complete AI-powered multi-camera surveillance system.

### 🎯 What You Accomplished

**👁️ SEE:**
- ✅ Connected 4 synchronized RTSP camera streams
- ✅ Previewed live feeds from multiple angles

**🧠 UNDERSTAND:**
- ✅ Indexed visual content across all cameras with AI
- ✅ Reviewed scene descriptions from different angles

**🎬 ACT:**
- ✅ Created 3 event detection rules
- ✅ Monitored real-time alerts via WebSocket
- ✅ Analyzed alert metrics by type and camera
- ✅ Generated synchronized multi-camera clips
- ✅ Composed a professional 2x2 grid video

---

### 🚀 What's Next?

Ready to build more advanced surveillance systems?

- 📖 **[VideoDB Documentation](https://docs.videodb.io)**: Complete API reference
- 🍳 **[VideoDB Cookbook](https://github.com/video-db/videodb-cookbook)**: More multicam examples
- 💬 **[Discord Community](https://discord.com/invite/py9P639jGz)**: Get help and share projects

---

**Build intelligent surveillance systems with VideoDB — the perception layer for real-world AI.** 🎥

